# Saved lists and pathway enrichment from a pairwise result

The app has three automations that do this by hand: **Pairwise Analysis Volcano**, **Pairwise
Analysis Generate Up/Down Regulated Lists**, and the **Reactome ORA Template**. You run them one
after the other, answering a form each time.

This notebook does the same chain from the API, with no clicking. It starts from a pairwise dataset
you already have:

1. pull the pairwise result table down and decide up and down regulated **in this script**
2. save those as entity lists in the workspace
3. build a tab with the volcanoes, a selection-driven table and an abundance plot
4. run a pathway enrichment job on each up-regulated list and plot the result on a second tab

Creating the pairwise dataset itself is a separate job and is covered by the pairwise walkthrough
notebook. Here you just need its id.

### The one thing that is genuinely different over the API

The in-app automation calls `PairwiseGetUpRegulated` and `PairwiseGetDownRegulated`. Those are
**browser instructions**. They read the volcano module's own thresholds and split the proteins
client-side. There is no endpoint for them, so you cannot ask the API "which proteins are up in this
volcano".

You do not need one. The pairwise result table is downloadable, it carries an adjusted p-value and
a log2 fold change per comparison, and the split is two lines of pandas. **When you script this, the
significance call is yours to make, in your own code, on data you can see.** That is arguably the
point: the threshold stops being a setting buried in a plot and becomes a line you can read, version
and change.

## Setup

In [ ]:
import io
import os

import pandas as pd
import requests
from dotenv import load_dotenv
from md_python import MDClient, Dataset

load_dotenv()
assert os.getenv("MD_AUTH_TOKEN"), "MD_AUTH_TOKEN is not set (check your .env)"

client = MDClient(version="v2")
print("API:", client.base_url)

## Configuration

Everything you would otherwise answer in three automation forms. One id and a handful of numbers.

In [ ]:
# The PAIRWISE dataset to work from. Make one with the pairwise walkthrough
# notebook or in the app, then paste its id here.
PAIRWISE_ID = "00000000-0000-0000-0000-000000000000"

ENTITY_TYPE = "protein"          # lowercase, always
CONDITION_COLUMN = "condition"   # the sample metadata column holding the groups

# Significance. The volcano module defaults to 0.05; we tighten it and, crucially,
# apply the SAME numbers here in Python when we build the lists, so the plot and
# the saved list agree.
FDR = 0.01
FOLD_CHANGE = 2              # linear, so |log2FC| >= 1

# Pathway enrichment. The ORA job takes a species name, not a taxon id.
# human | mouse | yeast | chinese_hamster
SPECIES = "human"

# Gene set database. Options depend on the species; for human they are Reactome,
# the three GO branches, and the MSigDB collections.
DATABASE = "Reactome"

TOP_N = 15                   # pathways shown in the dot plot

WORKSPACE_NAME = "Pairwise + ORA (API)"
ANALYSIS_TAB = "Differential expression"
ORA_TAB = "Pathway enrichment (ORA)"

## 1. Read the analysis off the pairwise dataset

A pairwise dataset already records what it was run on and what it compared, so there is nothing to
retype. Three things come straight off it:

* the **intensity dataset** it was computed from, which the entity lists and the abundance plot both
  need
* the **contrasts** it ran, as pairs of condition values
* through the intensity dataset, the **upload** and its sample metadata

Taking the contrasts from the job rather than typing them matters. Every downstream step looks up
columns named after the pair, like `Log2FC Treated - Control`. A pair typed by hand that does not
match the job exactly fails several cells later with a missing column, and the name it complains
about looks correct at a glance.

In [ ]:
import math

pairwise = client.datasets.get_by_id(PAIRWISE_ID)
assert pairwise, f"No dataset {PAIRWISE_ID!r}"
assert str(pairwise.type) == "PAIRWISE", f"{pairwise.name} is a {pairwise.type} dataset"
print(f"{pairwise.name}  ({pairwise.type}, {pairwise.state})")

# The intensity dataset this pairwise was computed from.
DATASET_ID = str(pairwise.input_dataset_ids[0])
dataset = client.datasets.get_by_id(DATASET_ID)
print(f"computed from: {dataset.name}  ({DATASET_ID})")

# The contrasts the job actually ran. (test, reference): positive log2FC means
# higher in the test group.
params = pairwise.job_run_params or {}
CONTRASTS = [tuple(p) for p in params["condition_comparisons"]["condition_comparison_pairs"]]
print(f"\n{len(CONTRASTS)} contrast(s):")
for left, right in CONTRASTS:
    print(f"   {left} vs {right}")

# The upload behind the intensity dataset, for its sample metadata. The groups
# are used by the abundance plot at the end.
upload_id = None
for u in client.uploads.query(search=dataset.name)["data"]:
    if DATASET_ID in [str(d.id) for d in client.datasets.list_by_upload(u["id"])]:
        upload_id = u["id"]
        break
assert upload_id, "Could not find the upload that owns the intensity dataset"

rows = client.uploads.get_sample_metadata(upload_id).data
header, body = rows[0], rows[1:]
groups = sorted({r[header.index(CONDITION_COLUMN)] for r in body})
print(f"\nupload {upload_id}\n{CONDITION_COLUMN}: {groups}")

## 2. Download the table and make the call yourself

`download_table_url` returns a presigned S3 URL for one table of the dataset. `output_comparisons`
is the one with the statistics.

Its columns are per contrast and named after the pair: `Log2FC Treated - Control`,
`AdjPValue Treated - Control`, and so on. Alongside them sit the identity columns every downstream
step needs: `ProteinIds`, `GeneNames`, `Description` and `GroupId`.

In [ ]:
url = client.datasets.download_table_url(PAIRWISE_ID, "output_comparisons")
raw = requests.get(url if isinstance(url, str) else url["url"]).content
comparisons = pd.read_csv(io.BytesIO(raw))

print(f"{len(comparisons)} entities x {len(comparisons.columns)} columns")
print("identity columns:", [c for c in ("ProteinIds", "GeneNames", "Description", "GroupId")
                            if c in comparisons.columns])
print("\nper-contrast columns:")
for left, right in CONTRASTS:
    pair = f"{left} - {right}"
    have = [f"{p} {pair}" in comparisons.columns for p in ("Log2FC", "AdjPValue")]
    print(f"  {pair:<28} Log2FC {have[0]}   AdjPValue {have[1]}")

In [ ]:
LOG2_FC = math.log2(FOLD_CHANGE)


def split_regulated(df, left, right, fdr=FDR, log2fc=LOG2_FC):
    """Up and down regulated entities for one contrast.

    This is the whole 'significance call', in the open. Change the comparison
    operators or the columns and you change what gets saved, with no hidden
    module setting involved.
    """
    pair = f"{left} - {right}"
    fc, p = df[f"Log2FC {pair}"], df[f"AdjPValue {pair}"]
    significant = p < fdr
    return (df[significant & (fc >= log2fc)].copy(),
            df[significant & (fc <= -log2fc)].copy())


regulated = {}
print(f"FDR < {FDR}, |log2FC| >= {LOG2_FC:g}  (fold change {FOLD_CHANGE}x)\n")
for left, right in CONTRASTS:
    up, down = split_regulated(comparisons, left, right)
    regulated[(left, right)] = {"Up": up, "Down": down}
    print(f"  {left} vs {right:<12} up {len(up):>4}   down {len(down):>4}")

## 3. Save them as entity lists

`client.workspaces.entity_lists.create()`. Lists live under a workspace, which is why the resource
hangs off `client.workspaces` rather than sitting at the top level of the client.

Each item needs an `entity_id`, a `group_id` and a `dataset_id`, which is exactly the `ProteinIds` /
`GroupId` pair from the table plus the dataset they belong to.

The non-obvious part is which dataset. It has to be the **intensity** dataset, not the pairwise one
the numbers came from: the server checks that every referenced dataset is `INTENSITY` and holds a
`Protein_Metadata` table, and a pairwise dataset is neither. Pass the pairwise id and you get a 400
that names the type. `GroupId` is safe to carry across, because a pairwise dataset inherits the
group ids of the intensity dataset it was computed from.

The names follow the automation's convention, `{left}_vs_{right}_Up_Reg`, so lists made this way sit
alongside lists made by hand in the app and sort together.

In [ ]:
workspace = client.workspaces.create(
    name=WORKSPACE_NAME, description="Pairwise, saved lists and ORA, built through the API"
)
WORKSPACE_ID = str(workspace.id)
print("workspace:", WORKSPACE_ID)


def create_entity_list(name, frame):
    """Create an entity list from rows of the comparisons table."""
    entity_list = client.workspaces.entity_lists.create(
        workspace_id=WORKSPACE_ID,
        name=name,
        entity_type=ENTITY_TYPE,
        items=[
            {"entity_id": str(r.ProteinIds), "group_id": int(r.GroupId), "dataset_id": DATASET_ID}
            for r in frame.itertuples()
        ],
    )
    return str(entity_list.id)


lists = {}
for (left, right), sides in regulated.items():
    for direction, frame in sides.items():
        if frame.empty:
            print(f"  skipped {left}_vs_{right}_{direction}_Reg (empty)")
            continue
        name = f"{left}_vs_{right}_{direction}_Reg"
        lists[(left, right, direction)] = create_entity_list(name, frame)
        print(f"  {name:<40} {len(frame):>4} entities")

print(f"\n{len(lists)} lists created")

## 4. The analysis tab

Three volcanoes, one per contrast, each carrying **its own** up and down list as a highlight. The
highlight shape comes from the automation's `SetProteinListModule` instruction: a list of
`{"id": ...}` per entity list.

The automation also offers `showAnnotation` with an `annotationLabel` of `"gene"`, and it is left
off here on purpose. These lists run to hundreds of proteins, and labelling every highlighted point
produces an unreadable plot. Switch it on in the app for a specific module once you have zoomed in.

`adjustedPValueThreshold` is set to the same `FDR` used to build the lists. If you change one and
not the other the plot and the saved list quietly disagree, which is the sort of thing nobody
notices until a reviewer does.

Under them go two selection-driven modules. Click points on a volcano and both follow:

* a **dataset table** with `source: "selection"`, showing only the identity columns
* an **entity abundance plot** on violin, with `conditionCol` set to the grouping column, so you see
  the per-sample intensities of the selected protein split by group

The table reads the intensity dataset's `Protein_Metadata`, deliberately not the pairwise
`output_comparisons`. Pointing a `dataset_table` at a pairwise table puts it in comparison mode,
where `conditionPairs` becomes required and the module refuses to render until you name a pair.
That is a different tool for a different job. Here we only want to know what got selected.

In [ ]:
PAIRWISE_SEARCH = {"individualResults": [{"id": PAIRWISE_ID, "name": pairwise.name}]}
INTENSITY_SEARCH = {"individualResults": [{"id": DATASET_ID, "name": dataset.name}]}


def highlight(left, right):
    """The up and down lists for this contrast, in SetProteinListModule's shape.

    No `showAnnotation`. The automation offers a gene-name label and it is
    tempting, but a list of a few hundred proteins puts a few hundred labels on
    the plot and you cannot read any of them. Turn it on per module in the app
    once you have zoomed into something worth naming.
    """
    return [
        {"id": lists[(left, right, d)]}
        for d in ("Down", "Up") if (left, right, d) in lists
    ]


def volcano(left, right):
    return {
        "datasetsSearch": PAIRWISE_SEARCH,
        "experimentAndConditionComparison": {
            "comparison": {"conditionPair": f"{left} - {right}", "left": left, "right": right}
        },
        "significantThresholdMethod": "fdr",
        "adjustedPValueThreshold": FDR,
        "foldChangeThreshold": FOLD_CHANGE,
        "proteinLists": highlight(left, right),
        "displayLegend": True,
    }


IDENTITY_COLUMNS = [{"name": c, "value": c} for c in ("ProteinIds", "GeneNames", "Description")]

ANALYSIS = [("heading", 0, 0, 12, 3, {"text": f"Differential expression, FDR < {FDR}"})]
for i, (left, right) in enumerate(CONTRASTS):
    ANALYSIS.append(("pairwise_volcano_plot", (i % 3) * 4, 3, 4, 18, volcano(left, right)))

ANALYSIS += [
    # Text modules take PLAIN TEXT. HTML tags are not allowed in a workspace.
    ("text", 0, 21, 12, 4, {"text": "Selected proteins\n"
                                    "Click points on any volcano above. "
                                    "Both modules below follow the selection."}),
    # Point this at the INTENSITY dataset's metadata table, not at the pairwise
    # output. A dataset_table over a pairwise table turns on the comparison-pair
    # machinery and then refuses to render until `conditionPairs` names at least
    # one pair, which is not what we want here: we want the identity of whatever
    # is selected, not its statistics.
    ("dataset_table", 0, 25, 12, 14, {
        "datasetsSearch": INTENSITY_SEARCH,
        "entityType": ENTITY_TYPE,
        "tableName": "Protein_Metadata",
        "source": "selection",
        "datasetTableValues": IDENTITY_COLUMNS,
        "selectableEntities": True,
    }),
    ("entity_abundance_plot", 0, 39, 12, 18, {
        "datasetsSearch": INTENSITY_SEARCH,
        "entityType": ENTITY_TYPE,
        "selectionType": "selection",
        "plotType": "violin",
        "conditionCol": CONDITION_COLUMN,
        "conditions": groups,
        "yScale": "intensity",
        "titleField": "protein_name",
        "numberOfColumns": 3,
    }),
]

tab = client.workspaces.tabs.create(workspace_id=WORKSPACE_ID, name=ANALYSIS_TAB)
ANALYSIS_TAB_ID = str(tab.id)

for item_id, x, y, w, h, settings in ANALYSIS:
    client.workspaces.modules.create_with_defaults(
        workspace_id=WORKSPACE_ID, tab_id=ANALYSIS_TAB_ID, item_id=item_id,
        x=x, y=y, width=w, height=h, settings=settings,
    )
    print(f"  ok  {item_id:<24} x={x:<3} y={y:<3} w={w:<3} h={h}")

## 5. Pathway enrichment: run the analyses, then plot them

The Reactome ORA template places three modules that each call out to Reactome live. That works, but
it is not an analysis you own: there is no dataset, nothing to download, and nothing to point a
second plot at.

There is a real ORA job on the platform, slug `ora`, and the API can run it. It takes an intensity
dataset, a foreground list of entity ids, a species and a gene-set database, and it produces an
`ENRICHMENT` dataset like any other job output. That dataset is then what the `ora_dot_plot` module
reads.

**Submit everything first, then wait.** The jobs run on the platform, not here, so they run
concurrently once submitted. Creating a job returns its dataset id immediately. If you submit and
wait one at a time, your wall clock is the sum of every job; if you submit them all and then wait,
it is the length of the slowest one. With three contrasts that is a nuisance, and with thirty it is
the difference between a coffee and an afternoon. The cell below is in three plain parts: submit
all, wait for all, then build the tab.

Two parameters are worth pausing on:

`foreground_ids` is a **flat list of entity ids**, not an entity list id. The app's field resolves
the list you pick down to its ids before submitting, so from a script you pass the accessions
straight through. We already have them in the dataframe.

`background` decides what the foreground is tested against. `"Detected features in this dataset"`
uses the proteins actually measured in this experiment, which is the honest choice for a
hypergeometric test. `"Selected Database"` uses everything in the database and will inflate your
enrichments, because the proteome you could not detect is not a fair reference universe.

In [ ]:
# --- 1. submit every job, waiting for none of them ---------------------------

submitted = []
for left, right in CONTRASTS:
    up = regulated[(left, right)]["Up"]
    if up.empty:
        print(f"  skipped  {left} vs {right}: no up-regulated entities")
        continue

    foreground = up["ProteinIds"].astype(str).tolist()
    ora_id = str(
        client.datasets.create(
            Dataset(
                input_dataset_ids=[DATASET_ID],
                name=f"ORA {left} vs {right} up ({DATABASE})",
                job_slug="ora",
                job_run_params={
                    "entity_type": ENTITY_TYPE,
                    "foreground_ids": foreground,
                    "species": SPECIES,
                    "database": DATABASE,
                    "background": "Detected features in this dataset",
                    "min_gene_set_size": 1,
                    "max_gene_set_size": 500,
                },
            )
        )
    )
    submitted.append((left, right, ora_id))
    print(f"  queued   {left} vs {right}: {len(foreground)} foreground -> {ora_id}")

print(f"\n{len(submitted)} job(s) running on the platform")


# --- 2. now wait for them all ------------------------------------------------

completed = []
for left, right, ora_id in submitted:
    state = client.datasets.wait_until_complete(
        upload_id=upload_id, dataset_id=ora_id, poll_s=10, timeout_s=1800
    )
    print(f"  {left} vs {right}: {state.state}")
    if str(state.state) == "COMPLETED":
        completed.append((left, right, ora_id))
    else:
        print(f"     {state.error_message}")


# --- 3. build the tab from whatever finished ---------------------------------

ora_tab = client.workspaces.tabs.create(workspace_id=WORKSPACE_ID, name=ORA_TAB)
ORA_TAB_ID = str(ora_tab.id)

y = 0
for left, right, ora_id in completed:
    ora = client.datasets.get_by_id(ora_id)
    client.workspaces.modules.create_with_defaults(
        workspace_id=WORKSPACE_ID, tab_id=ORA_TAB_ID, item_id="heading",
        x=0, y=y, width=12, height=2,
        settings={"text": f"{left} vs {right}, up-regulated, {DATABASE}"},
    )
    client.workspaces.modules.create_with_defaults(
        workspace_id=WORKSPACE_ID, tab_id=ORA_TAB_ID, item_id="ora_dot_plot",
        x=0, y=y + 2, width=12, height=16,
        settings={
            "datasetsSearch": {"individualResults": [{"id": ora_id, "name": ora.name}]},
            "topN": TOP_N,
            "xAxis": "GeneRatio",
            "entityListForSelection": "measured_in_dataset",
        },
    )
    print(f"  ok  {left} vs {right}")
    y += 18

print(f"\nOpen it: {client.base_url.replace('/api', '')}/w/{WORKSPACE_ID}/dashboard")

## What to take from this

**The threshold lives in your script, not in a plot.** `FDR` and `FOLD_CHANGE` are set once at the
top and used twice: to build the lists and to configure the volcanoes. Over the API that duplication
is visible and checkable. In the app the same two numbers live in a form and in a module's settings
panel, and nothing tells you when they drift apart.

**The lists are the join.** Once the up and down sets exist as entity lists they are ordinary
workspace objects: highlighted on the volcano, selectable in the app, and used as the foreground of
the ORA job. Nothing downstream re-derives them, so everything is looking at the same proteins.

**Enrichment is a job, not a widget.** Running `ora` gives you an `ENRICHMENT` dataset you can
download, diff between runs, and point more than one module at. The Reactome ORA modules call the
service live and leave nothing behind, which is fine for a look and poor for a pipeline.

**One trap worth knowing.** `entity_abundance_plot.conditions` is a required setting whose registry
default is an empty list. It validates clean and then renders empty, the same shape of problem as
the QC summary table's Table Grouping, so it is filled in explicitly above.